# H1 ABSA-RTS Pipeline (Colab v2)

This notebook runs the H1 negation/contrast diagnostics end-to-end with a repo sync step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/My Drive/AutoResearch/AutoResearch-Copilot')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Working directory:', PROJECT_ROOT)

## Sync latest code from GitHub

In [ ]:
!git --no-pager pull --ff-only

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'h1-negation-contrast-diagnostics' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)

## 1) Build ABSA-RTS (fixed negation templates)

In [ ]:
!PYTHONPATH=src python -m absa_rts.build_rts \
  --input data/absa_rts_seed_restaurants.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results \
  --domain restaurants

## 2) Generate A1 predictions (placeholder)

Replace this file later with your real A1 model outputs.

In [ ]:
NOISE = 0.10
SEED = 17

!PYTHONPATH=src python -m absa_rts.generate_predictions \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --output-csv experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --noise {NOISE} \
  --seed {SEED}

## 3) Evaluate A0 vs A1

In [ ]:
!PYTHONPATH=src python -m absa_rts.evaluate_baselines \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --pairs-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_pairs.csv \
  --a1-predictions experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results

In [ ]:
import json
from pathlib import Path

summary_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_summary.csv')
metrics_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_metrics.json')
print(summary_path.read_text(encoding='utf-8'))
print('\nDetailed metrics:\n')
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2))